In [0]:
source_path = "s3://databricks-s3-ingestion/sales_order/"

print(source_path)

s3://databricks-s3-ingestion/sales_order/


In [0]:
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option(
    "cloudFiles.schemaLocation",
    "s3://databricks-s3-ingestion/sales_order/_schema/"
)
    .load(source_path)
)

In [0]:
query = (
    df.writeStream
    .format("memory")
    .queryName("sales_orders_test")
    .outputMode("append")
     .trigger(availableNow=True)
    .start()
)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7585544705447235>, line 7
      1 query = (
      2     df.writeStream
      3     .format("memory")
      4     .queryName("sales_orders_test")
      5     .outputMode("append")
      6      .trigger(availableNow=True)
----> 7     .start()
      8 )

File <command-4643044561667282>, line 2
      1 df = (
----> 2     spark.readStream
      3     .format("cloudFiles")
      4     .option("cloudFiles.format", "csv")
      5     .option("header", "true")
      6     .option(
      7     "cloudFiles.schemaLocation",
      8     "s3://databricks-s3-ingestion/sales_order/_schema/"
      9 )
     10     .load(source_path)
     11 )

AnalysisException: [TEMP_CHECKPOINT_LOCATION_NOT_SUPPORTED] Implicit temporary streaming checkpoint locations are not supported in the current workspace, please specify a checkpoint location explicitl

In [0]:
display(df, checkpointLocation = "your_path")

Checkpointing to your_path


---------------------------------------------------------------------------
IllegalArgumentException                  Traceback (most recent call last)
File <command-6125555349490589>, line 1
----> 1 display(df, checkpointLocation = "your_path")

File /databricks/python_shell/lib/dbruntime/display.py:185, in Display.display(self, input, *args, **kwargs)
    183     pass
    184 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 185     self.display_connect_table(input, **kwargs)
    186 elif isinstance(input, ConnectDataFrame):
    187     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:112, in Display.display_connect_table(self, df, **kwargs)
    110 config = self._get_table_display_config()
    111 if df.isStreaming:
--> 112     self.cf_helper.display_streaming_dataframe(df, config, self.streaming_listener,
    113                                                **kwargs)
    114 else:
    115     self.cf_helper.display_datafram

In [0]:
query = (
    df.writeStream
    .format("memory")
    .queryName("sales_orders_test")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "s3://databricks-s3-ingestion/sales_order/_checkpoint/"
    )
    .trigger(availableNow=True)
    .start()
)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6125555349490590>, line 11
      1 query = (
      2     df.writeStream
      3     .format("memory")
      4     .queryName("sales_orders_test")
      5     .outputMode("append")
      6     .option(
      7         "checkpointLocation",
      8         "s3://databricks-s3-ingestion/sales_order/_checkpoint/"
      9     )
     10     .trigger(availableNow=True)
---> 11     .start()
     12 )

File <command-4643044561667282>, line 2
      1 df = (
----> 2     spark.readStream
      3     .format("cloudFiles")
      4     .option("cloudFiles.format", "csv")
      5     .option("header", "true")
      6     .option(
      7     "cloudFiles.schemaLocation",
      8     "s3://databricks-s3-ingestion/sales_order/_schema/"
      9 )
     10     .load(source_path)
     11 )

AnalysisException: This query does not support recoveri

In [0]:
source_path = "s3://databricks-s3-ingestion/sales_order/"

checkpoint_path = "s3://databricks-s3-ingestion/checkpoints/sales_order_stream/"

print("Source:", source_path)
print("Checkpoint:", checkpoint_path)

Source: s3://databricks-s3-ingestion/sales_order/
Checkpoint: s3://databricks-s3-ingestion/checkpoints/sales_order_stream/


In [0]:
df_bronze = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option(
        "cloudFiles.schemaLocation",
        "s3://databricks-s3-ingestion/schema/sales_order/"
    )
    .load(source_path)
)

In [0]:
df_bronze.printSchema()

---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File <command-6125555349490609>, line 1
----> 1 df_bronze.printSchema()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2076, in DataFrame.printSchema(self, level)
   2074     print(self.schema.treeString(level))
   2075 else:
-> 2076     print(self.schema.treeString())

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2049, in DataFrame.schema(self)
   2046 @property
   2047 def schema(self) -> StructType:
   2048     # self._schema call will cache the schema and serialize it if it is not cached yet.
-> 2049     _schema = self._schema
   2050     if self._cached_schema_serialized is not None:
   2051         try:

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2038, in DataFrame._schema(self)
   2036 if self._cached_schema

In [0]:
spark.sql("SHOW EXTERNAL LOCATIONS").show(truncate=False)

+-----------------------+----------------------------------------+-------+
|name                   |url                                     |comment|
+-----------------------+----------------------------------------+-------+
|salesorders_s3_location|s3://databricks-s3-ingestion/sales_order|NULL   |
+-----------------------+----------------------------------------+-------+



In [0]:
dbutils.fs.ls("s3://databricks-s3-ingestion/sales_order/")

[FileInfo(path='s3://databricks-s3-ingestion/sales_order/salesorders_001.csv', name='salesorders_001.csv', size=162743, modificationTime=1790090926000),
 FileInfo(path='s3://databricks-s3-ingestion/sales_order/_checkpoint/', name='_checkpoint/', size=0, modificationTime=1790179115735),
 FileInfo(path='s3://databricks-s3-ingestion/sales_order/_schema/', name='_schema/', size=0, modificationTime=1790179115735)]

In [0]:
df_bronze = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option(
        "cloudFiles.schemaLocation",
        "s3://databricks-s3-ingestion/sales_order/_schema/"
    )
    .load(source_path)
)

In [0]:
# 1. Define your S3 source paths
source_directory = "s3://your-bucket/raw-data-folder/"
schema_storage_directory = "s3://your-bucket/schema-folder/"  # Auto Loader will create the schema file here
checkpoint_directory = "s3://your-bucket/checkpoint-folder/"

# 2. Configure Auto Loader to automatically infer and store the schema
df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")                   # or "json", "parquet", etc.
    .option("cloudFiles.schemaLocation", schema_storage_directory)  # <--- Tells Auto Loader where to create the schema in S3
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")     # Automatically appends new columns if they appear
    .load(source_directory)
)

# 3. Write the stream to your target table
(df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_directory)
    .outputMode("append")
    .table("your_catalog.your_schema.target_table")

In [0]:
df_bronze.printSchema()

root
 |-- SALESORDERID: string (nullable = true)
 |-- CREATEDBY: string (nullable = true)
 |-- CREATEDAT: string (nullable = true)
 |-- CHANGEDBY: string (nullable = true)
 |-- CHANGEDAT: string (nullable = true)
 |-- FISCALYEARVARIANT: string (nullable = true)
 |-- FISCALYEARPERIOD: string (nullable = true)
 |-- NOTEID: string (nullable = true)
 |-- PARTNERID: string (nullable = true)
 |-- SALESORG: string (nullable = true)
 |-- CURRENCY: string (nullable = true)
 |-- GROSSAMOUNT: string (nullable = true)
 |-- NETAMOUNT: string (nullable = true)
 |-- TAXAMOUNT: string (nullable = true)
 |-- LIFECYCLESTATUS: string (nullable = true)
 |-- BILLINGSTATUS: string (nullable = true)
 |-- DELIVERYSTATUS: string (nullable = true)
 |-- _rescued_data: string (nullable = true)

